In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path().resolve()
auto_csv_path = cwd.parents[1] / "data" / "Auto.csv"

Basic prep before doing exercises (dropping nulls, indexing by name)

In [ ]:
auto = pd.read_csv(auto_csv_path, na_values=['?'])
auto[auto.isna().any(axis=1)]

In [ ]:
auto = auto.dropna()
auto = auto.set_index('name')
auto

In [ ]:
auto.describe()

- 9a. 
    - Quantitative predictors:
        - mpg, displacement, horsepower, weight, acceleration
    - Qualitative predictors:
        - cylinders, year, origin

In [ ]:
# count # of unique values in each column to see if they are quantitative / qualitative (this gives a good gauge, but should double check)
auto.nunique()

- 9b. Range of quantitative predictors (min / max)
- 9c. mean and std dev of each quantitative predictor (mean, std)

In [ ]:
auto[['mpg', 'displacement', 'horsepower', 'weight', 'acceleration']].agg(['min', 'max', 'mean', 'std'])

- 9d. Remove 10th to 85th observations & find range, mean and std dev
    - Observations: the summary statistics min, max, mean, std dev don't change very much, since the distribution of data is pretty random it's unlikely removing a small subset of data is going to change them much

In [ ]:
# use a mask to drop positions 10 to 85 (cannot use index values because of duplicates in the names)
mask = np.ones(len(auto), dtype=bool)
mask[10:86] = False
auto_10_85_dropped = auto.iloc[mask]
# auto_10_85_dropped

In [ ]:
auto_10_85_dropped[['mpg', 'displacement', 'horsepower', 'weight', 'acceleration']].agg(['min', 'max', 'mean', 'std'])

- 9e. Graphical investigation of predictors
    - acceleration looks normally distributed, with no clear relationship to the other quantitative variables
    - displacement, horsepower, and weight seem to be postively, linearly correlated
    - mpg seems to be inversely correlated with displacement, horsepower, and weight
- investigation of relationships w 'origin' and 'cylinders'
    - note that for 'cylinders' for values [3, 5] they may be unreliable because of the class imbalance (very few observations!)
    - origin: 1 (American) cars generally have higher displacement, higher horsepower, higher weight but lower mpg (which makes sense, they are less efficient I guess)
    - origin: 3 (Japanese) cars are most efficient with highest mpg - but could this be because of other factors, not because they are Japanese?
    - cylinders: obviously displacement is positively correlated with more cylinders, since it is a measure of total cylinder volume
    - cylinders: higher # of cylinders is associated with higher horsepower, higher weight, but lower mpg (seems like very heavy, powerful cars)
    - upon further investigation, the data only contains American (origin 1) cars for 8 cylinders. Hence, our prior analysis on origin is heavily skewed towards 8 cylinder characteristics for origin 1 cars.

In [ ]:
pd.plotting.scatter_matrix(auto[['mpg', 'displacement', 'horsepower', 'weight', 'acceleration']]);

In [ ]:
auto['origin'].value_counts().sort_index()

In [ ]:
auto['cylinders'].value_counts().sort_index()

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=5, figsize=(24, 5))
for col_idx, col in enumerate(['mpg', 'displacement', 'horsepower', 'weight', 'acceleration']):
    auto.boxplot(col, by='origin', ax=axs[col_idx])

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=5, figsize=(24, 5))
for col_idx, col in enumerate(['mpg', 'displacement', 'horsepower', 'weight', 'acceleration']):
    auto.boxplot(col, by='cylinders', ax=axs[col_idx])

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=5, figsize=(24, 5))
for col_idx, col in enumerate(['mpg', 'displacement', 'horsepower', 'weight', 'acceleration']):
    auto.boxplot(col, by=['origin', 'cylinders'], ax=axs[col_idx])

In [ ]:
# I asked Gemini to generate charts based on my previous code, but with color coding (using seaborn)
# Note to self: I should probably learn seaborn at some point!

import seaborn as sns
fig, axs = plt.subplots(nrows=1, ncols=5, figsize=(24, 5))
features = ['mpg', 'displacement', 'horsepower', 'weight', 'acceleration']

# Define custom palette for origins 1, 2, 3
custom_palette = {1: 'skyblue', 2: 'lightgreen', 3: 'tomato'}

for col_idx, col in enumerate(features):
    sns.boxplot(
        data=auto, 
        x='cylinders', 
        y=col, 
        hue='origin', 
        ax=axs[col_idx],
        palette=custom_palette
    )
    axs[col_idx].set_title(col)
    
    # # Optional: Clean up messy redundant legends across subplots
    # if col_idx > 0:
    #     axs[col_idx].get_legend().remove()

# plt.tight_layout()
plt.show()

- 9f. Prediction of mpg:
    - cylinders: it seems that having a higher number of cylinders leads to a lower mpg, but not for 3 cylinder cars
    - displacement, horsepower, weight: seems to have a 1/x relationship with mpg based on the scatterplot curve